In [ ]:
import pandas as pd
import altair as alt
import scipy.stats as stats

alt.data_transformers.disable_max_rows()

In [ ]:
original_data='./Data/final_tables/supplementary_file_1_BARD1_SGE_final_table.xlsx'
data_wd17='./Data/extra_data/20260723_BARD1.allscores_wD17.tsv'

In [ ]:
original_df = pd.read_excel(original_data, sheet_name='scores')
wd17_df=pd.read_csv(data_wd17, sep='\t')

wd17_df = wd17_df.loc[wd17_df['ref'].str.len()==1]
wd17_df['pos_id'] = wd17_df['pos'].astype(str) + ':' + wd17_df['alt']

# Inspect Scores for All Regions

## Initial Data Processing

In [ ]:
og_nod17 = original_df[['target', 'pos_id', 'functional_consequence', 'score']]
wd17_nod17=wd17_df[['target', 'pos_id', 'functional_consequence', 'score']]

wd17_nod17= wd17_nod17.rename(columns={'score': 'wd17_score'})

In [ ]:
merged_no_d17 = pd.merge(og_nod17, wd17_nod17, on=['target', 'pos_id'], how='inner')
merged_no_d17

## Scatter Plot

In [ ]:
non_d17_scatter = alt.Chart(merged_no_d17).mark_point().encode(
    x = 'score:Q',
    y='wd17_score:Q',
).facet('target:N', columns=5)

non_d17_scatter.display()

## Correlation Heatmap

In [ ]:
grouped = merged_no_d17.groupby('target')

output_tuples = []
for group, target_df in grouped:
    og_score=target_df['score']
    wd17_score=target_df['wd17_score']

    corr, _ = stats.pearsonr(og_score, wd17_score)

    return_tuple = (group, corr, 'orignal vs. w_d17')
    output_tuples.append(return_tuple)

corr_df = pd.DataFrame(output_tuples, columns=['target', 'correlation', 'test'])
print(corr_df)

In [ ]:
corr_heatmap = alt.Chart(corr_df).mark_rect().encode(
    x='test:N',
    y='target:N',
    color='correlation:Q',
    tooltip=['correlation']
)

corr_heatmap.display()

In [ ]:
print(merged_no_d17.value_counts('functional_consequence_x').reset_index())
print(merged_no_d17.value_counts('functional_consequence_y').reset_index())
